# Thêm dữ liệu TMDB vào Dataset

In [ ]:
!pip install gdown requests pandas tqdm

import gdown
import pandas as pd
import requests
from tqdm import tqdm
import numpy as np

links = pd.read_csv("links.csv")
movies = pd.read_csv("movies.csv")

print("Links DataFrame Head:")
print(links.head())
print("\nMovies DataFrame Head:")
print(movies.head())

API_KEY = "d2cd3d8689e233058695b271e56fe7c7"
BASE_URL = "https://api.themoviedb.org/3/movie/"

def get_tmdb_info(tmdb_id):
    """Trả về overview, keywords, production company theo tmdb_id."""

    if tmdb_id is None or pd.isna(tmdb_id):
        return pd.Series([None, None, None])

    try:
        url = f"{BASE_URL}{int(tmdb_id)}?api_key={API_KEY}&append_to_response=keywords"
        response = requests.get(url)

        if response.status_code != 200:
            return pd.Series([None, None, None])

        data = response.json()

        overview = data.get("overview") or None

        keywords = data.get("keywords", {})

        if "keywords" in keywords:
            keyword_list = [kw["name"] for kw in keywords["keywords"]]
        elif "results" in keywords:
            keyword_list = [kw["name"] for kw in keywords["results"]]
        else:
            keyword_list = []

        keyword_str = ", ".join(keyword_list) if keyword_list else None

        companies = data.get("production_companies", [])
        company = companies[0]["name"] if companies else None

        return pd.Series([overview, keyword_str, company])

    except Exception:
        return pd.Series([None, None, None])


merged = pd.merge(
    movies,
    links[['movieId', 'tmdbId']],
    left_on='ID',
    right_on='movieId',
    how='left'
)

tqdm.pandas()
merged[['Overview', 'Keyword', 'Network']] = merged['tmdbId'].progress_apply(get_tmdb_info)

merged.to_csv("movies.csv", index=False)
print("\n✅ Đã thêm Overview, Keyword, Network vào 'movies.csv'!")
print(merged[['Title', 'Overview', 'Keyword', 'Network']].head())


Links DataFrame Head:
   movieId  imdbId   tmdbId
0        1  114709    862.0
1        2  113497   8844.0
2        3  113228  15602.0
3        4  114885  31357.0
4        5  113041  11862.0

Movies DataFrame Head:
   ID                               Title                        Genres
0   1                    Toy Story (1995)   Animation|Children's|Comedy
1   2                      Jumanji (1995)  Adventure|Children's|Fantasy
2   3             Grumpier Old Men (1995)                Comedy|Romance
3   4            Waiting to Exhale (1995)                  Comedy|Drama
4   5  Father of the Bride Part II (1995)                        Comedy


100%|██████████| 3883/3883 [06:51<00:00,  9.43it/s]


✅ Đã thêm Overview, Keyword, Network vào 'movies.csv'!
                                Title  \
0                    Toy Story (1995)   
1                      Jumanji (1995)   
2             Grumpier Old Men (1995)   
3            Waiting to Exhale (1995)   
4  Father of the Bride Part II (1995)   

                                            Overview  \
0  Led by Woody, Andy's toys live happily in his ...   
1  When siblings Judy and Peter discover an encha...   
2  A family wedding reignites the ancient feud be...   
3  Cheated on, mistreated and stepped on, the wom...   
4  Just when George Banks has recovered from his ...   

                                             Keyword              Network  
0  rescue, friendship, mission, jealousy, villain...                Pixar  
1  giant insect, board game, disappearance, jungl...     TriStar Pictures  
2  fishing, sequel, old man, best friend, wedding...       Lancaster Gate  
3  based on novel or book, single mother, divorce...    

# Kiểm tra tổng số lượng dữ liệu mỗi bảng

In [ ]:
print(f"Number of users: {len(users_df)}")
print(f"Number of movies: {len(movies_df)}")
print(f"Number of ratings {len(ratings_df)}")

Number of users: 6040
Number of movies: 3883
Number of ratings 1000209


# Tiền xử lý dữ liệu

In [ ]:
!pip install pandas numpy tqdm

import pandas as pd
import numpy as np
from tqdm import tqdm

users_df = pd.read_csv("users.csv")   
movies_df = pd.read_csv("movies.csv") 
ratings_df = pd.read_csv("ratings.csv") 

print("Users Head:")
print(users_df.head())
print("\nMovies Head:")
print(movies_df.head())
print("\nRatings Head:")
print(ratings_df.head())

def is_numeric(x):
    try:
        float(x)
        return True
    except:
        return False

users_df.dropna(inplace=True)
users_df.drop_duplicates(inplace=True)
users_df = users_df[users_df['UserID'].apply(is_numeric)]
users_df = users_df[users_df['Gender'].isin(['M', 'F'])]
valid_ages = ['Under 18', '18-24', '25-34', '35-44', '45-49', '50-55', '56+']
users_df = users_df[users_df['Age'].isin(valid_ages)]

movies_df.dropna(inplace=True)
movies_df.drop_duplicates(inplace=True)
movies_df = movies_df[movies_df['ID'].apply(is_numeric)]
movies_df = movies_df[
    (movies_df['Title'].str.strip() != "") &
    (movies_df['Genres'].str.strip() != "")
].reset_index(drop=True)

ratings_df.dropna(inplace=True)                
ratings_df.drop_duplicates(inplace=True)       

ratings_df = ratings_df[
    (ratings_df['UserID'].apply(is_numeric)) &
    (ratings_df['MovieID'].apply(is_numeric)) &
    (ratings_df['Rating'].apply(is_numeric))
]

ratings_df = ratings_df[(ratings_df['Rating'] >= 1) & (ratings_df['Rating'] <= 5)]

valid_movie_ids = movies_df['ID'].astype(int).tolist()
ratings_df = ratings_df[ratings_df['MovieID'].astype(int).isin(valid_movie_ids)]

print("\nUsers:", users_df.shape)
print("Movies:", movies_df.shape)
print("Ratings:", ratings_df.shape)
print("\nUsers columns:", users_df.columns)
print("Movies columns:", movies_df.columns)
print("Ratings columns:", ratings_df.columns)

users_df.to_csv("users.csv", index=False)
movies_df.to_csv("movies.csv", index=False)
ratings_df.to_csv("ratings.csv", index=False)

print("\n✔️ Đã lưu dữ liệu sạch cho 3 file users, movies và ratings thành công!")


Users Head:
   UserID Gender       Age            Occupation Zip-code
0       1      F  Under 18          K-12 student    48067
1       2      M       56+         self-employed    70072
2       3      M     25-34             scientist    55117
3       4      M     45-49  executive/managerial    02460
4       5      M     25-34                writer    55455

Movies Head:
   ID                               Title                        Genres  \
0   1                    Toy Story (1995)   Animation|Children's|Comedy   
1   2                      Jumanji (1995)  Adventure|Children's|Fantasy   
2   3             Grumpier Old Men (1995)                Comedy|Romance   
3   4            Waiting to Exhale (1995)                  Comedy|Drama   
4   5  Father of the Bride Part II (1995)                        Comedy   

   movieId   tmdbId                                           Overview  \
0      1.0    862.0  Led by Woody, Andy's toys live happily in his ...   
1      2.0   8844.0  When s

# Chia train / test cho file Ratings

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

ratings = pd.read_csv("ratings.csv") 

def split_userwise(df, test_size=0.2, seed=42):
    train_list = []
    test_list = []

    for user_id, group in df.groupby('UserID'):
        if len(group) < 2:
            train_list.append(group)
            continue

        train, test = train_test_split(group, test_size=test_size, random_state=seed)
        train_list.append(train)
        test_list.append(test)

    train_df = pd.concat(train_list).reset_index(drop=True)
    test_df = pd.concat(test_list).reset_index(drop=True)
    return train_df, test_df

train_80, test_20 = split_userwise(ratings, test_size=0.2)
train_80.to_csv("train_ratings_80.csv", index=False)
test_20.to_csv("test_ratings_20.csv", index=False)

train_70, test_30 = split_userwise(ratings, test_size=0.3)
train_70.to_csv("train_ratings_70.csv", index=False)
test_30.to_csv("test_ratings_30.csv", index=False)

print("Hoàn tất chia dữ liệu!")


Hoàn tất chia dữ liệu!


# Tạo User profile từ Ratings và User actions

In [ ]:
import pandas as pd
import numpy as np

ratings = pd.read_csv("ratings.csv")           
user_actions = pd.read_csv("user_actions.csv")  

rating_to_action = {
    1: 6,  
    2: 7,  
    3: 8,  
    4: 9,   
    5: 10   
}

all_action_ids = list(range(1, 34))  

exclude = [6, 7, 8, 9, 10]  
extra_actions = [a for a in all_action_ids if a not in exclude]


def random_extra_actions():
    """Sinh 1–4 action phụ."""
    k = np.random.randint(1, 5)
    return list(np.random.choice(extra_actions, k, replace=False))

new_rows = []

for _, row in ratings.iterrows():
    uid = int(row["UserID"])
    mid = int(row["MovieID"])
    rating = int(row["Rating"])

    new_rows.append({
        "UserID": uid,
        "MovieID": mid,
        "ActionID": rating_to_action[rating]
    })

    for action in random_extra_actions():
        new_rows.append({
            "UserID": uid,
            "MovieID": mid,
            "ActionID": action
        })

final_profile = pd.DataFrame(new_rows)

final_profile.drop_duplicates(inplace=True)

final_profile.to_csv("user_profiles.csv", index=False)

print("DONE! File user_profiles.csv đã được tạo!")
final_profile.head()


DONE! File user_profiles.csv đã được tạo!


,UserID,MovieID,ActionID
0,1,1193,10
1,1,1193,11
2,1,661,8
3,1,661,4
4,1,661,3


# Lưu file

In [ ]:
import os
for file in ['user_profiles.csv', 'movies.csv', 'users.csv']:
    print(f"{file}: {'Tồn tại' if os.path.exists(file) else 'Không tìm thấy'}")
from google.colab import files
files.download('user_profiles.csv')
files.download('movies.csv')
files.download('users.csv')

user_profiles.csv: Tồn tại
movies.csv: Tồn tại
users.csv: Tồn tại


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>